**BPE tokenizer from scratch**

Build a working Byte-Pair Encoding tokenizer to understand how GPT-style tokenization works

In [ ]:
import re
from collections import Counter


class SimpleBPETokenizer:
  # a minimal bpe tokenizer that shows how exactly the pairs get merged
  def __init__(self):
    self.vocab = {} # token -> id
    self.reverse_vocab = {} # id -> token
    self.pairs = [] # list of pairs to be merged
    self.merges = [] # list to store merged pairs


  def get_pairs(self, word):
    pairs = Counter()
    chars = list(word)
    for i in range(len(chars) - 1):
      pairs[(chars[i], chars[i+1])] += 1
    return pairs

  def train(self, text, vocab_size=50):
    # train bpe, repeatedly merge most common pairs
    words = text.split()
    # add </w> at the end of the word marker like gpt
    word_freqs = Counter(''.join(list(w) + ['</w>']) for w in words)

    # intitalize vocab with all characters
    all_chars = set(''.join(word_freqs.keys()))
    self.vocab = {ch: i for i, ch in enumerate(sorted(all_chars))}
    self.reverse_vocab = {i:ch  for ch, i in self.vocab.items()}

    print(f"Starting vocab: {len(self.vocab)} characters")
    print(f"Training to vocab size: {vocab_size}")
    print('-' * 40)

    # BPE merges most frequent pairs
    while len(self.vocab) < vocab_size:
      # count all pairs across all words
      pair_counts = Counter()
      for word, freq in word_freqs.items():
        pairs = self.get_pairs(word)
        for pair, count in pairs.items():
          pair_counts[pair] += count * freq
      if not pair_counts:
        break

    # find and merge most frequent pair
    best_pair = pair_counts.most_common(1)[0][0]
    new_token = best_pair[0] + best_pair[1]

    print(f"Merge #{len(self.merges)+1}: "
                  f"'{best_pair[0]}' + '{best_pair[1]}' -> '{new_token}'")
    new_id = len(self.vocab)
    self.vocab[new_token] = new_id
    self.reverse_vocab[new_id] = new_token
    self.merges.append((best_pair, new_token))

    # apply merge to all words
    new_word_freqs = {}
    for word, freq in word_freqs.items():
      new_word = word.replace(
          best_pair[0] + best_pair[1], new_token
      )
      new_word_freqs[new_word] = freq
    word_freqs = new_word_freqs

    print('-' * 40)
    print(f'Final vocab size: {len(self.vocab)}')

  def tokenize(self, text):
    # tokenize text using learned merges
    words = text.split()
    all_tokens = []

    for word in words:
      tokens = list(word) + ['</w>']

      # Apply merges iteratively
      for pair, new_token in self.merges:
        i = 0
        while i < len(tokens) - 1:
          if (tokens[i], tokens[i+1]) == pair:
            tokens = tokens[:i] + [new_token] + tokens[i+2:]
          else:
            i += 1
      all_tokens.extend(tokens)
    return all_tokens

  def encode(self, text):
    tokens = self.tokenize(text)
    return [self.vocab.get(t, 0) for t in tokens]

  def decode(self, ids):
    tokens = [self.reverse_vocab.get(i, '?') for i in ids]
    text = ''.join(tokens).replace('</w>', ' ')
    return text.strip()

corpus = """
the cat sat on the mat
the dog sat on the log
the cat and the dog played
"""

print("=" * 50)
print("BPE TOKENIZER TRAINING")
print("=" * 50)

tokenizer = SimpleBPETokenizer()
tokenizer.train(corpus, vocab_size=30)


for text in ["the cat", "the dog sat", "cat and dog"]:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    print(f"\nText: '{text}'")
    print(f"  Tokens: {tokens}")
    print(f"  IDs:    {ids}")
    print(f"  Decoded: '{tokenizer.decode(ids)}'")

BPE TOKENIZER TRAINING
Starting vocab: 18 characters
Training to vocab size: 30
----------------------------------------
